# 01 — Nettoyage des données

**Entrée** : `data/raw/owid-co2-data.csv` (jeu de données CO₂ d'Our World in Data)  
**Sortie** : `data/processed/co2_clean.csv`

Étapes : suppression des lignes sans code ISO et des agrégats (monde, continents, groupes de revenus), restriction à 1900–2024, sélection des colonnes utiles, suppression des lignes sans valeur de `co2`.

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/owid-co2-data.csv')
print("Shape brut :", df.shape)
df.head()

Shape brut : (50411, 79)


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
df = df[df['iso_code'].notna()]
print("Après filtre iso_code :", df.shape)

Après filtre iso_code : (42480, 79)


In [3]:
aggregates = [
    'World', 'Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania',
    'European Union (27)', 'European Union (28)', 'High-income countries',
    'Low-income countries', 'Lower-middle-income countries', 'Upper-middle-income countries',
    'International transport', 'International aviation', 'International shipping',
]

df = df[~df['country'].isin(aggregates)]
df = df[~df['iso_code'].astype(str).str.startswith('OWID_', na=False)]
print("Après exclusion agrégats :", df.shape)

Après exclusion agrégats : (42480, 79)


In [4]:
df = df[(df['year'] >= 1900) & (df['year'] <= 2024)]
print("Après filtre années :", df.shape)
print("Période :", df['year'].min(), '-', df['year'].max())

Après filtre années : (27250, 79)
Période : 1900 - 2024


In [5]:
key_cols = [
    'country', 'iso_code', 'year', 'population', 'gdp',
    'co2', 'co2_per_capita', 'cumulative_co2', 'co2_growth_prct',
    'coal_co2', 'oil_co2', 'gas_co2', 'land_use_change_co2',
    'methane', 'nitrous_oxide',
    'temperature_change_from_co2', 'temperature_change_from_ghg',
]

df = df[key_cols]
print("Après sélection colonnes :", df.shape)
df.head()

Après sélection colonnes : (27250, 17)


,country,iso_code,year,population,gdp,co2,co2_per_capita,cumulative_co2,co2_growth_prct,coal_co2,oil_co2,gas_co2,land_use_change_co2,methane,nitrous_oxide,temperature_change_from_co2,temperature_change_from_ghg
150,Afghanistan,AFG,1900,4707744.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.670736,5.051904,0.481894,0.000120,0.000204
151,Afghanistan,AFG,1901,4751177.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.720090,5.087836,0.484724,0.000124,0.000209
152,Afghanistan,AFG,1902,4802500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.755704,5.123855,0.488086,0.000127,0.000215
153,Afghanistan,AFG,1903,4861833.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.783001,5.159859,0.491899,0.000130,0.000221
154,Afghanistan,AFG,1904,4921891.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.805755,5.195950,0.496078,0.000134,0.000226


In [6]:
missing_pct = (df.isna().sum() / len(df) * 100).round(1).sort_values(ascending=False)
print(missing_pct)

gas_co2                        57.0
gdp                            49.5
coal_co2                       45.3
oil_co2                        30.4
co2_growth_prct                28.8
co2_per_capita                 28.2
cumulative_co2                 27.4
co2                            27.4
land_use_change_co2             9.6
methane                         8.7
nitrous_oxide                   7.8
population                      4.3
temperature_change_from_co2     1.4
temperature_change_from_ghg     1.4
country                         0.0
year                            0.0
iso_code                        0.0
dtype: float64

In [7]:
print("Avant suppression NaN :", df.shape)
df_clean = df.dropna(subset=['co2']).copy()
print("Après suppression NaN :", df_clean.shape)

Avant suppression NaN : (27250, 17)
Après suppression NaN : (19787, 17)


In [ ]:
df_clean.to_csv('../data/processed/co2_clean.csv', index=False)
print("✅ Fichier exporté : ../data/processed/co2_clean.csv", df_clean.shape)


✅ Fichier exporté : co2_clean.csv (19787, 17)
